# Synchronize KB against the real database

`PipelineService.synchronize_kb()` is the backfill for the knowledge base: it finds every
`EnrichedRecord` in the database that has no `document_kb` row yet -- because the KB was
down when it was enriched, because indexing failed, or because it predates the KB feature
-- and indexes each one. It is the method behind `POST /knowledge/synchronize-kb`.

Unlike `knowledge_indexing.ipynb` (which uses a throwaway SQLite file so its demo writes
never touch real data), **this notebook targets your actual `data/classiflow.db` and your
actual `data/chroma` collection directly -- no throwaway database, no fabricated records.**
It reads whatever `EnrichedRecord` rows are really sitting unindexed in your dev database
and indexes them for real.

> **Kernel**: select the project's `.venv` kernel in the top-right picker.
>
> **This writes permanently to `data/classiflow.db` and `data/chroma/chroma.sqlite3`.**
> There is no cleanup section -- the whole point is to leave the real catalogue and vector
> store caught up. Section 2 shows you exactly which records are about to be indexed
> *before* section 3 writes anything, so you can stop there if that is not what you want.
>
> **First run downloads a model**: `paraphrase-multilingual-MiniLM-L12-v2` is ~470 MB,
> same as `knowledge_indexing.ipynb`.
>
> **Requires migrations up to date**: `enriched_records.filename`/`sha256` and the
> `document_kb` table were added by alembic revisions 0009-0011. Run
> `uv run alembic upgrade head` first if you have not already.

## 1 -- Connect to the real database and build the real indexing services

`Settings.DATABASE_URL` defaults to a *relative* path (`./data/classiflow.db`), which
breaks with "unable to open database file" when the kernel's cwd is not the repo root.
Anchored to the package location instead, same fix `pipeline_end_to_end.ipynb` uses --
but here it points at the existing file, nothing is deleted or recreated.

`Settings.CHROMA_PATH` is already an absolute path by default
(`data/chroma`, set from `_PROJECT_ROOT` in `settings.py`), so `ChromaVectorStore()`
needs no adjustment to reach the real collection.

In [4]:
from pathlib import Path

from sqlalchemy.ext.asyncio import async_sessionmaker, create_async_engine

import classiflow
from classiflow.knowledge.chunking.chunker import ChunkerService
from classiflow.knowledge.embeddings.embedder import SentenceTransformerEmbedder
from classiflow.knowledge.indexing.csv_metadata import CsvDocumentMetadataRepository
from classiflow.knowledge.indexing.indexer import IndexerService
from classiflow.knowledge.vectordb.chroma_store import ChromaVectorStore
from classiflow.settings import Settings

_db_path = Path(classiflow.__file__).parents[2] / "data" / "classiflow.db"
assert _db_path.exists(), f'{_db_path} not found -- run "uv run alembic upgrade head" first.'
Settings.DATABASE_URL = f"sqlite+aiosqlite:///{_db_path.as_posix()}"

engine = create_async_engine(Settings.DATABASE_URL, echo=False)
session_factory = async_sessionmaker(engine, expire_on_commit=False)

chunker = ChunkerService()
embedder = SentenceTransformerEmbedder()
metadata_repo = CsvDocumentMetadataRepository()
store = ChromaVectorStore()
indexer = IndexerService(
    chunker=chunker, embedder=embedder, vector_store=store, metadata_repo=metadata_repo
)

print(f"database: {_db_path}")
print(f"chroma  : {Settings.chroma_path}")
print(f"chroma collection count (before): {store.count()}")

c:\Repos\Diplo-TP-Final\Trabajo-Integrador\Trabajo-Integrador\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


database: C:\Repos\Diplo-TP-Final\Trabajo-Integrador\Trabajo-Integrador\data\classiflow.db
chroma  : ./data/chroma
chroma collection count (before): 40


## 2 -- Preview what is about to be indexed

`find_unindexed()` runs the real `NOT EXISTS` query against `document_kb`
(`SqlEnrichedRecordRepository`, no fabricated data). Nothing is written yet -- this is
read-only, so it is safe to just look.

In [5]:
from classiflow.database.repositories.enriched_record import SqlEnrichedRecordRepository

async with session_factory() as session:
    pending = await SqlEnrichedRecordRepository(session).find_unindexed()

print(f"unindexed EnrichedRecords: {len(pending)}")
for record in pending:
    print(
        f"  id={record.id:<5} job_id={record.job_id:<40} "
        f"filename={record.filename!r:<30} sha256={record.sha256!r}"
    )

unindexed EnrichedRecords: 22
  id=1     job_id=c3793cda-229f-4659-bd23-548e5d94aa8e     filename=None                           sha256=None
  id=2     job_id=20a2d2b7-5d85-4fcd-8629-03678a2b4387     filename=None                           sha256=None
  id=3     job_id=2c1ae559-e2aa-4975-ad9f-a4f6a4f2e3d7     filename=None                           sha256=None
  id=4     job_id=01c929da-66ab-473f-850a-f58efa99cde8     filename=None                           sha256=None
  id=5     job_id=a65e8b63-ec0e-4977-8154-52d7700ee9b4     filename=None                           sha256=None
  id=6     job_id=1d5e22e8-9e10-46a4-a2dc-a70c2ed8d034     filename=None                           sha256=None
  id=7     job_id=8d7f92c7-4a90-4548-a841-946e76c717b9     filename=None                           sha256=None
  id=8     job_id=adba8e2d-c889-4542-a07f-cc50c86e2c7f     filename=None                           sha256=None
  id=9     job_id=e43f55b6-6dd6-410b-9d34-728fd765cd1a     filename=None          

### 2a -- Records with no `filename`/`sha256` are indexed under an empty hash

`filename` and `sha256` were added to `enriched_records` by migration 0011, after this
table already had rows -- those existing rows have both columns `NULL`. `synchronize_kb`
falls back to `record.filename or ""` / `record.sha256 or ""` for them
(`PipelineService.synchronize_kb`, `service.py`), which still indexes their
`cleaned_text`, but under a chunk id derived from an empty hash and with no CSV metadata
match (no real filename to resolve). This is expected for that older cohort, not a bug --
flagged here so the counts in section 4 are not a surprise.

In [6]:
_missing_identity = [r for r in pending if not r.filename or not r.sha256]
print(f"pending records missing filename/sha256: {len(_missing_identity)} / {len(pending)}")

pending records missing filename/sha256: 22 / 22


## 3 -- Run `synchronize_kb()`

`PipelineService` takes 11 constructor dependencies; only `enriched_record_repo`,
`indexer` and `document_kb_repo` matter for this method, so the rest are `None` --
exactly how `tests/shared/test_pipeline_service_kb_sync.py` builds it. This is the cell
that actually writes: real chunks into `data/chroma`, real rows into `document_kb`.

In [7]:
import asyncio

from classiflow.database.repositories.document_kb import SqlDocumentKbRepository
from classiflow.domain.repositories.document_kb import IDocumentKbRepository
from classiflow.domain.repositories.enriched_record import IEnrichedRecordRepository
from classiflow.services.pipeline.service import PipelineService


def build_service(
    enriched_repo: IEnrichedRecordRepository, kb_repo: IDocumentKbRepository
) -> PipelineService:
    return PipelineService(
        job_repo=None,  # type: ignore[arg-type]  # unused by synchronize_kb
        document_steps_repo=None,  # type: ignore[arg-type]
        enriched_record_repo=enriched_repo,
        broadcaster=None,  # type: ignore[arg-type]
        coordinator=None,  # type: ignore[arg-type]
        enrichment_coordinator=None,  # type: ignore[arg-type]
        document_storage=None,  # type: ignore[arg-type]
        classification_coordinator=None,  # type: ignore[arg-type]
        job_semaphore=asyncio.Semaphore(1),  # unused
        indexer=indexer,
        document_kb_repo=kb_repo,
    )


async with session_factory() as session:
    service = build_service(SqlEnrichedRecordRepository(session), SqlDocumentKbRepository(session))
    indexed_job_ids, skipped = await service.synchronize_kb()
    await session.commit()

print(f"indexed: {len(indexed_job_ids)}")
for job_id in indexed_job_ids:
    print(f"  {job_id}")
print(f"skipped: {skipped}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4761.63it/s]


indexed: 22
  c3793cda-229f-4659-bd23-548e5d94aa8e
  20a2d2b7-5d85-4fcd-8629-03678a2b4387
  2c1ae559-e2aa-4975-ad9f-a4f6a4f2e3d7
  01c929da-66ab-473f-850a-f58efa99cde8
  a65e8b63-ec0e-4977-8154-52d7700ee9b4
  1d5e22e8-9e10-46a4-a2dc-a70c2ed8d034
  8d7f92c7-4a90-4548-a841-946e76c717b9
  adba8e2d-c889-4542-a07f-cc50c86e2c7f
  e43f55b6-6dd6-410b-9d34-728fd765cd1a
  9f8853d4-1cb4-4fbe-9077-045357591c77
  c4125ade-b711-425d-9d09-3e9a9f8cfb14
  9d007b2c-703d-4a20-9d4a-82dbf5b11534
  84701900-5bd4-428d-8268-f182d44ab2fd
  305b078f-dd24-4686-acd1-bca1aca7ce2c
  a42ca62f-98fa-4e0f-a8fd-e95e651e567d
  2a517973-2a88-4920-b289-75e60679f037
  c288fefc-3baa-477f-90c6-da4d9cee92e8
  8450270d-f04d-48de-92b0-f29e719ce747
  303152ba-48a7-4d29-87fe-64a70ce0979b
  890d763e-82e1-447e-997a-4697cfadfee4
  4b48dbf7-365d-4f8e-9479-49043c3a1b95
  9651d70a-4a5b-4686-875f-c89cf3f3af16
skipped: 0


## 4 -- Verify against `document_kb` and the Chroma collection

Confirms the rows are really there and the collection count grew by the number of
chunks actually written -- the source of truth check, not just trusting the return value
from section 3.

In [8]:
from sqlalchemy import select

from classiflow.database.models import DocumentKb

async with session_factory() as session:
    rows = (
        (await session.execute(select(DocumentKb).where(DocumentKb.job_id.in_(indexed_job_ids))))
        .scalars()
        .all()
    )

print(f"{'job_id':<40} {'filename':<30} chunk_count")
for row in rows:
    print(f"{row.job_id:<40} {row.filename:<30} {row.chunk_count}")

print(f"\nchroma collection count (after): {store.count()}")
assert len(rows) == len(indexed_job_ids)

await engine.dispose()

job_id                                   filename                       chunk_count
01c929da-66ab-473f-850a-f58efa99cde8                                    3
1d5e22e8-9e10-46a4-a2dc-a70c2ed8d034                                    2
20a2d2b7-5d85-4fcd-8629-03678a2b4387                                    36
2a517973-2a88-4920-b289-75e60679f037                                    3
2c1ae559-e2aa-4975-ad9f-a4f6a4f2e3d7                                    18
303152ba-48a7-4d29-87fe-64a70ce0979b                                    23
305b078f-dd24-4686-acd1-bca1aca7ce2c                                    14
4b48dbf7-365d-4f8e-9479-49043c3a1b95                                    9
8450270d-f04d-48de-92b0-f29e719ce747                                    8
84701900-5bd4-428d-8268-f182d44ab2fd                                    3
890d763e-82e1-447e-997a-4697cfadfee4                                    4
8d7f92c7-4a90-4548-a841-946e76c717b9                                    7
9651d70a-4a5b-4686-875f-